<a href="https://colab.research.google.com/github/chrliem/amazon-delivery-perfomance/blob/main/Optimizing_Amazon_Delivery_Perfomance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Optimizing Amazon Delivery Perfomance**

Christian Darma Setiawan

**Problem Statement:** Delivery to customer is a crucial component of e-commerce logistics. Any problem during this stage can significantly drive up the operational costs. External factors such as weather conditions, traffic density, and agent performance often cause delays. Therefore, companies must optimize these delivery processes to ensure strict adherence to SLA.

**Objective**:
To identify factors contributing to delivery delays through operational data analysis and to build a predictive model for accurate delivery time estimation.

**Analytic Questions**
1. Does the distance between the store and the customer significantly impact delivery duration compared to other variables?
2. How does the combination of weather conditions and traffic density contribute to delivery delays?
3. Is there a correlation between an agent's age or rating and the operational efficiency they attain in the field?
4. What is the average duration for the key fulfillment stages (Order-to-Pickup vs. Pickup-to-Delivery)?
5. Are there specific time windows where delivery performance significantly drops compared to the baseline?
6. How do different vehicle types compare in performance when navigating various environmental factors?
7. How accurate is the predictive model in estimating delivery times across different scenarios?

In [1]:
import pandas as pd
import requests
import time
import os
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## **Data Overview**

In [3]:
base_path = '/content/drive/MyDrive/'
file_name = 'amazon_delivery.csv'
full_path = os.path.join(base_path, file_name)

print("--- Loading Dataset ---")

if os.path.exists(full_path):
    df = pd.read_csv(full_path)

    date_cols = [col for col in df.columns if 'date' in col.lower()]
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')

    print(f"Rows: {len(df):>8,} | Cols: {len(df.columns)}")
    print("\n" + "="*50 + "\n")

    display(df.head())
else:
    print(f"Error: {file_name} tidak ditemukan di {full_path}!")


--- Loading Dataset ---
Rows:   43,739 | Cols: 16




,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys


In [4]:
df.shape

(43739, 16)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43739 entries, 0 to 43738
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Order_ID         43739 non-null  object        
 1   Agent_Age        43739 non-null  int64         
 2   Agent_Rating     43685 non-null  float64       
 3   Store_Latitude   43739 non-null  float64       
 4   Store_Longitude  43739 non-null  float64       
 5   Drop_Latitude    43739 non-null  float64       
 6   Drop_Longitude   43739 non-null  float64       
 7   Order_Date       43739 non-null  datetime64[ns]
 8   Order_Time       43739 non-null  object        
 9   Pickup_Time      43739 non-null  object        
 10  Weather          43648 non-null  object        
 11  Traffic          43739 non-null  object        
 12  Vehicle          43739 non-null  object        
 13  Area             43739 non-null  object        
 14  Delivery_Time    43739 non-null  int64

## **Data Quality Assesment**

In [6]:
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df)) * 100

null_summary = pd.DataFrame({
    'Total Null': null_counts,
    'Percentage (%)': null_pct.round(2)
}).filter(items = null_counts[null_counts > 0].index, axis=0).sort_values('Total Null', ascending=False)

print("--- Missing Values Audit ---")
if not null_summary.empty:
    display(null_summary)
else:
    print("No missing values found.")

duplicate_count = df.duplicated().sum()
print(f"\nTotal Duplicate Rows: {duplicate_count}")

--- Missing Values Audit ---


,Total Null,Percentage (%)
Weather,91,0.21
Agent_Rating,54,0.12



Total Duplicate Rows: 0


In [7]:
df.describe()

,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Delivery_Time
count,43739.000000,43685.000000,43739.000000,43739.000000,43739.000000,43739.000000,43739,43739.000000
mean,29.567137,4.633780,17.210960,70.661177,17.459031,70.821842,2022-03-13 15:58:10.697089792,124.905645
min,15.000000,1.000000,-30.902872,-88.366217,0.010000,0.010000,2022-02-11 00:00:00,10.000000
25%,25.000000,4.500000,12.933298,73.170283,12.985996,73.280000,2022-03-04 00:00:00,90.000000
50%,30.000000,4.700000,18.551440,75.898497,18.633626,76.002574,2022-03-15 00:00:00,125.000000
75%,35.000000,4.900000,22.732225,78.045359,22.785049,78.104095,2022-03-27 00:00:00,160.000000
max,50.000000,6.000000,30.914057,88.433452,31.054057,88.563452,2022-04-06 00:00:00,270.000000
std,5.815155,0.334716,7.764225,21.475005,7.342950,21.153148,NaN,51.915451


In [8]:
print("--- Categorical Columns Checking ---")
cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    unique_vals = df[col].unique()
    num_unique = df[col].nunique()
    print(f"\nColumn: {col} ({num_unique} unique values)")
    print(f"Values: {unique_vals}")

--- Categorical Columns Checking ---

Column: Order_ID (43739 unique values)
Values: ['ialx566343618' 'akqg208421122' 'njpu434582536' ... 'xnek760674819'
 'cynl434665991' 'nsyz997960170']

Column: Order_Time (177 unique values)
Values: ['11:30:00' '19:45:00' '08:30:00' '18:00:00' '13:30:00' '21:20:00'
 '19:15:00' '17:25:00' '20:55:00' '21:55:00' '14:55:00' '17:30:00'
 '09:20:00' '19:50:00' '20:25:00' '20:30:00' '20:40:00' '21:15:00'
 '20:20:00' '22:30:00' '08:15:00' '19:30:00' '12:25:00' '18:35:00'
 '20:35:00' '23:20:00' '23:35:00' '22:35:00' '23:25:00' '13:35:00'
 '21:35:00' '18:55:00' '14:15:00' '11:00:00' '09:45:00' '08:40:00'
 '23:00:00' '19:10:00' '10:55:00' '21:40:00' '19:00:00' '16:45:00'
 '15:10:00' '22:45:00' '22:10:00' '20:45:00' '22:50:00' '17:55:00'
 '09:25:00' '20:15:00' '22:25:00' '22:40:00' '23:50:00' '15:25:00'
 '10:20:00' '10:40:00' '15:55:00' '20:10:00' '12:10:00' '15:30:00'
 '10:35:00' '21:10:00' '20:50:00' '12:35:00' '21:00:00' '23:40:00'
 '18:15:00' '18:20:00' '11:

In [9]:
rating_anomalies = df[df['Agent_Rating'] > 5]

print(f"Rating >5: {len(rating_anomalies)} rows")

Rating >5: 53 rows


In [10]:
geo_anomalies = df[
    (df['Store_Latitude'] < 1.0) |
    (df['Store_Longitude'] < 1.0) |
    (df['Drop_Latitude'] < 1.0) |
    (df['Drop_Longitude'] < 1.0)
]

print(f"Unusual Lat/Long (< 1.0): {len(geo_anomalies)} rows")

Unusual Lat/Long (< 1.0): 3693 rows


In [11]:
print("\nMin Lat Long:")
print(df[['Store_Latitude', 'Store_Longitude', 'Drop_Latitude', 'Drop_Longitude']].min())


Min Lat Long:
Store_Latitude    -30.902872
Store_Longitude   -88.366217
Drop_Latitude       0.010000
Drop_Longitude      0.010000
dtype: float64


## **Data Cleaning**

In [12]:
df_clean = df[df['Agent_Rating'] <= 5].copy()

In [13]:
df_clean = df_clean[
    (df_clean['Store_Latitude'] > 1.0) &
    (df_clean['Store_Longitude'] > 1.0) &
    (df_clean['Drop_Latitude'] > 1.0) &
    (df_clean['Drop_Longitude'] > 1.0)
]

In [14]:
cols_to_strip = ['Traffic', 'Vehicle', 'Area']
for col in cols_to_strip:
    df_clean[col] = df_clean[col].str.strip()

In [15]:
df_clean.replace('NaN', np.nan, inplace=True)
df_clean.dropna(subset=['Weather', 'Traffic', 'Agent_Rating'], inplace=True)

In [16]:
print(f"Original data: {len(df)} rows")
print(f"Cleaned data: {len(df_clean)} rows")
print(f"Removed data pct: {((len(df) - len(df_clean)) / len(df) * 100):.2f}%")

Original data: 43739 rows
Cleaned data: 39958 rows
Removed data pct: 8.64%


In [17]:
df_clean.describe()

,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Delivery_Time
count,39958.000000,39958.000000,39958.000000,39958.000000,39958.000000,39958.000000,39958,39958.000000
mean,29.554332,4.634003,18.897525,76.909839,18.961162,76.973475,2022-03-14 07:32:44.602833152,125.107338
min,20.000000,2.500000,9.957144,72.768726,9.967144,72.778726,2022-02-11 00:00:00,10.000000
25%,25.000000,4.500000,12.986047,73.897902,13.065479,73.939315,2022-03-05 00:00:00,90.000000
50%,30.000000,4.700000,19.065838,76.618203,19.115838,76.662278,2022-03-15 00:00:00,125.000000
75%,35.000000,4.900000,22.751234,78.347554,22.818060,78.399645,2022-03-27 00:00:00,160.000000
max,39.000000,5.000000,30.914057,88.433452,31.054057,88.563452,2022-04-06 00:00:00,270.000000
std,5.761347,0.314920,5.463903,3.489784,5.465724,3.490006,NaN,51.946315


## **Feature Engineering**

In [18]:
def haversine(lat1, lon1, lat2, lon2):
    r = 6371

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return r * c

In [19]:
df_clean['distance_km'] = haversine(
    df_clean['Store_Latitude'], df_clean['Store_Longitude'],
    df_clean['Drop_Latitude'], df_clean['Drop_Longitude']
)

In [20]:
df_clean.head()

,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category,distance_km
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing,3.025149
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics,20.183530
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports,1.552758
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics,7.790401
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys,6.210138


In [21]:
df_clean['Order_Datetime'] = pd.to_datetime(df_clean['Order_Date'].dt.date.astype(str) + ' ' + df_clean['Order_Time'])
df_clean['Pickup_Datetime'] = pd.to_datetime(df_clean['Order_Date'].dt.date.astype(str) + ' ' + df_clean['Pickup_Time'])

mask = df_clean['Pickup_Datetime'] < df_clean['Order_Datetime']
df_clean.loc[mask, 'Pickup_Datetime'] = df_clean.loc[mask, 'Pickup_Datetime'] + pd.Timedelta(days=1)

df_clean['prep_time_minutes'] = (df_clean['Pickup_Datetime'] - df_clean['Order_Datetime']).dt.total_seconds() / 60

anomalies_after_fix = df_clean[df_clean['prep_time_minutes'] < 0]
print(f"Negative prep time: {len(anomalies_after_fix)}")

Negative prep time: 0


In [22]:
df_clean.describe()

,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Delivery_Time,distance_km,Order_Datetime,Pickup_Datetime,prep_time_minutes
count,39958.000000,39958.000000,39958.000000,39958.000000,39958.000000,39958.000000,39958,39958.000000,39958.000000,39958,39958,39958.000000
mean,29.554332,4.634003,18.897525,76.909839,18.961162,76.973475,2022-03-14 07:32:44.602833152,125.107338,9.716000,2022-03-15 01:27:35.555833600,2022-03-15 01:37:34.452174848,9.981606
min,20.000000,2.500000,9.957144,72.768726,9.967144,72.778726,2022-02-11 00:00:00,10.000000,1.465067,2022-02-11 00:00:00,2022-02-11 00:05:00,5.000000
25%,25.000000,4.500000,12.986047,73.897902,13.065479,73.939315,2022-03-05 00:00:00,90.000000,4.657655,2022-03-05 17:35:00,2022-03-05 17:45:00,5.000000
50%,30.000000,4.700000,19.065838,76.618203,19.115838,76.662278,2022-03-15 00:00:00,125.000000,9.193021,2022-03-15 23:00:00,2022-03-15 23:10:00,10.000000
75%,35.000000,4.900000,22.751234,78.347554,22.818060,78.399645,2022-03-27 00:00:00,160.000000,13.631449,2022-03-27 19:55:00,2022-03-27 20:08:45,15.000000
max,39.000000,5.000000,30.914057,88.433452,31.054057,88.563452,2022-04-06 00:00:00,270.000000,20.969489,2022-04-06 23:55:00,2022-04-07 00:05:00,15.000000
std,5.761347,0.314920,5.463903,3.489784,5.465724,3.490006,NaN,51.946315,5.598498,NaN,NaN,4.086348


In [23]:
df_clean['order_hour'] = df_clean['Order_Datetime'].dt.hour
df_clean['day_of_week'] = df_clean['Order_Datetime'].dt.dayofweek

def time_slot(hour):
    if 5 <= hour < 12: return 'Morning'
    elif 12 <= hour < 17: return 'Afternoon'
    elif 17 <= hour < 21: return 'Evening'
    else: return 'Night'

df_clean['time_slot'] = df_clean['order_hour'].apply(time_slot)

In [24]:
df_clean.head()

,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,...,Area,Delivery_Time,Category,distance_km,Order_Datetime,Pickup_Datetime,prep_time_minutes,order_hour,day_of_week,time_slot
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,...,Urban,120,Clothing,3.025149,2022-03-19 11:30:00,2022-03-19 11:45:00,15.0,11,5,Morning
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,...,Metropolitian,165,Electronics,20.183530,2022-03-25 19:45:00,2022-03-25 19:50:00,5.0,19,4,Evening
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,...,Urban,130,Sports,1.552758,2022-03-19 08:30:00,2022-03-19 08:45:00,15.0,8,5,Morning
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,...,Metropolitian,105,Cosmetics,7.790401,2022-04-05 18:00:00,2022-04-05 18:10:00,10.0,18,1,Evening
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,...,Metropolitian,150,Toys,6.210138,2022-03-26 13:30:00,2022-03-26 13:45:00,15.0,13,5,Afternoon


In [25]:
weather_risk = {
    'Sunny': 1, 'Cloudy': 2, 'Windy': 3,
    'Fog': 4, 'Sandstorms': 5, 'Stormy': 5
}
df_clean['weather_risk_score'] = df_clean['Weather'].map(weather_risk)

In [26]:
df_clean['is_weekend'] = df_clean['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

df_clean[['day_of_week', 'is_weekend']].head()

,day_of_week,is_weekend
0,5,1
1,4,0
2,5,1
3,1,0
4,5,1


In [27]:
df_clean.to_csv('amazon_delivery_cleaned_final.csv', index=False)

## EDA

1. Does the distance between the store and the customer significantly impact delivery duration compared to other variables?

The result show a correlation cofficient of 0.28 between delivery distance and duration (delivery distance calculated using haversine) which indicating a weak to moderate positive relation. While the line shows a trend where delivery time increases with distance, the high dispersion of data points suggest that distance alone is not the primamry driver of delays.

External factors like traffic, weather, and/or agent perfomance exert influence on the total delivery time rather than the distance alone.

In [28]:
correlation = df_clean['distance_km'].corr(df_clean['Delivery_Time'])

fig1 = px.scatter(
    df_clean,
    x='distance_km',
    y='Delivery_Time',
    trendline='ols',
    title=f'Impact of Distance on Delivery Duration (Correlation: {correlation:.2f})',
    labels={'distance_km': 'Distance (km)', 'Delivery_Time': 'Duration (min)'},
    opacity=0.5,
    template='plotly_white'
)

fig1.show()

2. How does the combination of weather conditions and traffic density contribute to delivery delays?

From the heatmap, there is a significant impact from weather conditions and traffic density on delivery time. The most severe delays are observed when traffic 'Jam' with 'Cloudy' or 'Foggy' weatherm wuth average delivery times peaking at approx 174 minutes. In contrast, 'Sunny' weather appears to act as a mitigating factor with the lowest average durations around 96-110 minutes even when traffic density is not at its lowest.
The need of dynamic buffer time during delivery scheduling might be an important thing.

In [29]:
weather_traffic_avg = df_clean.groupby(['Weather', 'Traffic'])['Delivery_Time'].mean().reset_index()

heatmap_data = weather_traffic_avg.pivot(index='Weather', columns='Traffic', values='Delivery_Time')

fig2 = px.imshow(
    heatmap_data,
    text_auto='.1f',
    aspect="auto",
    color_continuous_scale='YlOrRd',
    title='Average Delivery Time: Interaction between Weather and Traffic',
    labels=dict(x="Traffic Density", y="Weather Condition", color="Avg Time (min)"),
    template='plotly_white'
)

fig2.show()

3. Is there a correlation between an agent's age or rating and the operational efficiency they attain in the field?

The result below indicates that each agent characteristics (age and rating) do not significantly impact the operational efficiency.
The scatter plot shows a slight uptrend, indicating that delivery speed increase slightly as the agent's age. But from the data points where they are scattered between 50-250 minutes accorss all age, agent's age is not the primary factor that influence the delivery time. It means that individual perfomance in this case have less influence if compared to external factor in previous analysis.

In [30]:
fig3a = px.box(
    df_clean,
    x='Agent_Rating',
    y='Delivery_Time',
    title='Delivery Time Distribution by Agent Rating',
    color='Agent_Rating',
    template='plotly_white'
)

fig3b = px.scatter(
    df_clean,
    x='Agent_Age',
    y='Delivery_Time',
    trendline='ols',
    title='Correlation: Agent Age vs Delivery Time',
    labels={'Agent_Age': 'Age of Agent', 'Delivery_Time': 'Duration (min)'},
    opacity=0.3,
    template='plotly_white'
)

fig3a.show()
fig3b.show()

4. What is the average duration for the key fulfillment stages?

The result shows that preparationg stage only representing 7.39% of total time with average 9.98 minutes, while the travel time accounting 92.5% with average 125 minutes.

In [31]:
import plotly.graph_objects as go

avg_prep = df_clean['prep_time_minutes'].mean()
avg_travel = df_clean['Delivery_Time'].mean()

fig4 = go.Figure(data=[
    go.Bar(name='Preparation (Order to Pickup)', x=['Logistics Stages'], y=[avg_prep], marker_color='indianred'),
    go.Bar(name='Travel (Pickup to Delivery)', x=['Logistics Stages'], y=[avg_travel], marker_color='lightsalmon')
])

fig4.update_layout(
    barmode='stack',
    title='Average Delivery Stage Breakdown',
    yaxis_title='Minutes',
    template='plotly_white'
)

fig4.show()


In [32]:
fig4_pie = px.pie(
    values=[avg_prep, avg_travel],
    names=['Prep Time', 'Travel Time'],
    title='Time Distribution Percentage',
    hole=0.4
)
fig4_pie.show()

5. Are there specific time windows where delivery performance significantly drops compared to the baseline?

The baseline average is 125 mintes. Significant drops shows during two primary window, between 11 AM - 2 PM and 5 PM to 9 PM, where average durations reach the maximum at 150 minutes.
These delivery time might be caused by traffic during peak hour with the combination of high order volume. Early morning and late night windows show the best perfomance.

In [33]:
hourly_trend = df_clean.groupby('order_hour')['Delivery_Time'].mean().reset_index()

fig5 = px.line(
    hourly_trend,
    x='order_hour',
    y='Delivery_Time',
    markers=True,
    title='Average Delivery Duration by Hour of Day',
    labels={'order_hour': 'Hour of Day (24h)', 'Delivery_Time': 'Avg Duration (min)'},
    template='plotly_white'
)
fig5.add_hline(y=df_clean['Delivery_Time'].mean(), line_dash="dash", line_color="red", annotation_text="Global Average")
fig5.show()

6. How do different vehicle types compare in performance when navigating various environmental factors?

The result shows that motorcycles exhibit the highest average deluvery time accross all traffic situation where it peaked at 155 minutes. In contrast, scooters and vans shows slighly better perfomance with average around 134 and 137 mins in traffic jam.

In [34]:
fig6 = px.bar(
    df_clean.groupby(['Vehicle', 'Traffic'])['Delivery_Time'].mean().reset_index(),
    x='Vehicle',
    y='Delivery_Time',
    color='Traffic',
    barmode='group',
    title='Vehicle Efficiency Across Different Traffic Densities',
    labels={'Delivery_Time': 'Avg Duration (min)'},
    template='plotly_white'
)
fig6.show()

## Modeling

In [35]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00


In [36]:
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

In [37]:
features = ['Agent_Age', 'Agent_Rating', 'distance_km', 'prep_time_minutes',
            'is_weekend', 'order_hour', 'Weather', 'Traffic', 'Vehicle', 'Area', 'Category']
cat_cols = ['Weather', 'Traffic', 'Vehicle', 'Area', 'Category']

X = df_clean[features].copy()
y = df_clean['Delivery_Time']

In [38]:
X_encoded = X.copy()
le = LabelEncoder()
for col in cat_cols:
    X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))

In [39]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)
X_train_cat, X_test_cat, _, _ = train_test_split(X, y, test_size=0.2, random_state=42)

In [40]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [41]:
xgb_model = XGBRegressor(n_estimators=1000, learning_rate=0.1, max_depth=6, random_state=42)
xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=None, num_parallel_tree=None, ...)

In [42]:
cat_model = CatBoostRegressor(iterations=1000, learning_rate=0.1, depth=6,
                              cat_features=cat_cols, verbose=0, random_state=42)
cat_model.fit(X_train_cat, y_train)

CatBoostRegressor(cat_features=['Weather', 'Traffic', 'Vehicle', 'Area', 'Category'], depth=6, iterations=1000, learning_rate=0.1, loss_function='RMSE', random_state=42, verbose=0)

In [43]:
models = {
    'Random Forest': rf_model.predict(X_test),
    'XGBoost': xgb_model.predict(X_test),
    'CatBoost': cat_model.predict(X_test_cat)
}

results = []
for name, pred in models.items():
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    results.append({'Model': name, 'MAE (Minutes)': round(mae, 2), 'R2 Score': round(r2, 2)})

In [44]:
results_df = pd.DataFrame(results).sort_values(by='MAE (Minutes)')
results_df

,Model,MAE (Minutes),R2 Score
2,CatBoost,17.16,0.82
0,Random Forest,17.86,0.80
1,XGBoost,18.07,0.80


The result from 3 predictive models shows that CatBoost delivers the highest precision for estimating delivery durations. With a MAE of 17.16 minutes and R2 score 0.82. The catboost captures 82% of the variance in the datase and outperforming random forest and xgboost that likely due to catboost advanced gradient boosting algorithm which optimized for the categorical compelxities. In practical terms, this degree of catboost accuracy means that the model's predictions typically deviate by only about 17 minutes from the actual delivery time

In [45]:
fig_compare = px.bar(results_df, x='Model', y='MAE (Minutes)', color='Model',
                     title='Model Comparison',
                     text_auto=True, template='plotly_white')
fig_compare.show()

In [46]:
scenario_data = {
    'Agent_Age': 28,
    'Agent_Rating': 4.8,
    'distance_km': 12.5,
    'prep_time_minutes': 15,
    'is_weekend': 0,
    'order_hour': 18,
    'Weather': 'Stormy',
    'Traffic': 'Jam',
    'Vehicle': 'motorcycle',
    'Area': 'Urban',
    'Category': 'Electronics'
}

test_case = pd.DataFrame([scenario_data])

prediction = cat_model.predict(test_case)[0]

print("="*50)
print("MODEL INFERENCE TEST")
print("="*50)
print(f"Order Category   : {scenario_data['Category']}")
print(f"Order Time       : {scenario_data['order_hour']}:00")
print(f"Day Type         : {'Weekend' if scenario_data['is_weekend'] == 1 else 'Weekday'}")
print("-" * 30)
print(f"Distance         : {scenario_data['distance_km']} km")
print(f"Weather Condition: {scenario_data['Weather']}")
print(f"Traffic Density  : {scenario_data['Traffic']}")
print(f"Vehicle Type     : {scenario_data['Vehicle']}")
print("-" * 30)
print(f"Agent Rating     : {scenario_data['Agent_Rating']} / 5.0")
print(f"Prep Time        : {scenario_data['prep_time_minutes']} minutes")
print("="*50)
print(f"PREDICTED DELIVERY TIME: {prediction:.2f} MINUTES")
print("="*50)

MODEL INFERENCE TEST
Order Category   : Electronics
Order Time       : 18:00
Day Type         : Weekday
------------------------------
Distance         : 12.5 km
Weather Condition: Stormy
Traffic Density  : Jam
Vehicle Type     : motorcycle
------------------------------
Agent Rating     : 4.8 / 5.0
Prep Time        : 15 minutes
PREDICTED DELIVERY TIME: 121.76 MINUTES
